# Fabric MLV file ingestion demo: files → bronze → silver → gold
Spark SQL cells. Run top to bottom, following README.md for the upload/refresh order. Every statement cites the Microsoft Learn page or blog it was taken from.

## 0. Setup
Attach this notebook to a **schema-enabled lakehouse** (Fabric Runtime 1.3). Before running anything, find/replace `<workspace>` and `<lakehouse>` in the bronze cell, or paste the ABFS path of `Files/mlv_demo/table_a/` (Lakehouse explorer > folder `...` > Copy ABFS path). Upload **only `file_001.csv`** to `Files/mlv_demo/table_a/` first.

In [ ]:
%%sql
-- 00_setup_schemas.sql
-- Run in a Spark SQL notebook attached to a schema-enabled lakehouse.
-- Prereq "Lakehouse with Lakehouse schemas enabled and Fabric Runtime 1.3":
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/get-started-with-materialized-lake-views
-- CREATE SCHEMA IF NOT EXISTS pattern: same Get Started page.
-- Schema names must NOT be all-uppercase (Current limitations):
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/create-materialized-lake-view#current-limitations

CREATE SCHEMA IF NOT EXISTS bronze;

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS gold;

## 1. Bronze: file-ingesting MLV (`USING OneLake_Files`, DYNAMIC, APPEND_ONLY)

In [ ]:
%%sql
-- 01_bronze_table_a.sql
-- Bronze: ingest every CSV in Files/mlv_demo/table_a/ straight into a materialized lake view.
-- Syntax copied from the "Ingest files with USING OneLake_Files" example:
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/create-materialized-lake-view#ingest-files-with-using-onelake_files
-- and the announcement blog (same statement, DYNAMIC + APPEND_ONLY):
--   https://community.fabric.microsoft.com/blog/fbc_fabricupdatesblogs/file-data-ingestion-with-materialized-lake-views-in-microsoft-fabric-preview/5367801
--
-- Replace <workspace> and <lakehouse> (in demo_ws: demo_ws and demo_mlv_lh), or paste the folder's
--   "Copy ABFS path" from the Lakehouse explorer.
-- Path rules (Spark SQL reference): trailing slash for a folder; no raw spaces or %20 in the path.
-- schema_mode DYNAMIC: "adds newly discovered columns and writes NULL when a file doesn't contain an
--   established column" (TBLPROPERTIES reference). This is what makes file_003's attribute_b appear.
-- refresh_mode APPEND_ONLY: "adds rows from new files" (TBLPROPERTIES reference).
-- No AS SELECT here on purpose: projecting a fixed column list would hide new columns.
-- The doc example states "Each row also gets a __filepath__ column" for this exact form, and the blog's
--   GROUP BY __filepath__ query runs against it. (See README "Doc uncertainties" #1.)
-- Do NOT set the Spark CSV 'mode' option: "Fabric manages malformed-row handling" (OPTIONS reference).

CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS bronze.table_a
USING OneLake_Files
OPTIONS (
    'format' = 'csv',
    'path'   = 'abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Files/mlv_demo/table_a/',
    'header' = 'true'
)
COMMENT "Bronze: raw table_a CSVs ingested from OneLake Files (demo)"
TBLPROPERTIES (
    'schema_mode'  = 'DYNAMIC',
    'refresh_mode' = 'APPEND_ONLY'
);

## 2. Silver: typed + validated (`CONSTRAINT ... ON MISMATCH DROP`)

In [ ]:
%%sql
-- 02_silver_table_a_clean.sql  (v1 - created after file_001, before attribute_b exists)
-- Table-based MLV on top of the file-ingesting bronze view.
-- CREATE ... (CONSTRAINT name CHECK (cond) ON MISMATCH DROP) ... COMMENT ... TBLPROPERTIES ... AS SELECT
--   clause order from the Syntax block:
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/create-materialized-lake-view#syntax
-- ON MISMATCH DROP "silently removes the row"; default is FAIL; "The lineage view shows the count of
--   dropped records":
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/data-quality
-- Constraints use only deterministic built-ins so incremental refresh stays possible:
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/refresh-materialized-lake-view#sql-constructs-supported-by-incremental-refresh
-- TBLPROPERTIES (delta.enableChangeDataFeed=true) at create time is documented on the refresh page
--   ("You can enable CDF at creation time"); it lets gold (which reads silver) refresh incrementally.
-- Selecting __filepath__ from bronze downstream: UNVERIFIED in docs (see README uncertainties #1).
-- CASTs are defensive: the default of 'inferSchema' for OneLake_Files is not documented (uncertainty #3).

CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS silver.table_a_clean
(
    CONSTRAINT id_not_null         CHECK (id IS NOT NULL)              ON MISMATCH DROP,
    CONSTRAINT non_negative_value  CHECK (value >= 0)                  ON MISMATCH DROP,
    CONSTRAINT known_segment       CHECK (segment IN ('A', 'B', 'C'))  ON MISMATCH DROP
)
COMMENT "Silver: typed, validated table_a rows (rows breaking a rule are dropped)"
TBLPROPERTIES (delta.enableChangeDataFeed=true)
AS SELECT
    CAST(id AS INT)                AS id,
    segment,
    attribute_a,
    CAST(value AS DECIMAL(10,2))   AS value,
    __filepath__                   AS source_file
FROM bronze.table_a;

## 3. Gold: aggregate

In [ ]:
%%sql
-- 03_gold_value_by_segment.sql
-- Gold aggregate, same shape as the blog's gold example (SUM ... GROUP BY):
--   https://community.fabric.microsoft.com/blog/fbc_fabricupdatesblogs/file-data-ingestion-with-materialized-lake-views-in-microsoft-fabric-preview/5367801
-- IF NOT EXISTS + AS SELECT ... GROUP BY form: Get Started page
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/get-started-with-materialized-lake-views
-- Only SUM/COUNT are used: they "support incremental refresh without the partitioning requirement"
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/refresh-materialized-lake-view#sql-constructs-supported-by-incremental-refresh

CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS gold.value_by_segment
COMMENT "Gold: row count and total value per segment"
AS SELECT
    segment,
    COUNT(*)    AS row_count,
    SUM(value)  AS total_value
FROM silver.table_a_clean
GROUP BY segment;

## 4. Inspect (re-run this section after every refresh)
Expected numbers per step are in `expected/EXPECTED.md`. Cell (e) only returns `attribute_b` after file_003; before that it errors with an unresolved column, which is expected.

In [ ]:
%%sql
-- 04_inspect.sql  - run after every refresh. Plain SELECTs against MLVs (they query like Delta tables:
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/overview-materialized-lake-view )

-- (a) Rows per source file. Query taken from the blog ("reconcile row counts back to individual source files"):
--   https://community.fabric.microsoft.com/blog/fbc_fabricupdatesblogs/file-data-ingestion-with-materialized-lake-views-in-microsoft-fabric-preview/5367801
--   and the GROUP BY __filepath__ tip on the lineage page:
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/view-lineage#interpret-file-processing-metrics
SELECT __filepath__, COUNT(*) AS rows
FROM bronze.table_a
GROUP BY __filepath__;
-- (b) Bronze vs silver per file = dropped rows per file. __filepath__ is a full OneLake path

In [ ]:
%%sql
--   element_at(split(...)) is a standard Spark SQL built-in used only for readability.
SELECT element_at(split(b.__filepath__, '/'), -1) AS file,
       b.bronze_rows,
       COALESCE(s.silver_rows, 0)                 AS silver_rows,
       b.bronze_rows - COALESCE(s.silver_rows, 0) AS dropped_rows
FROM (SELECT __filepath__, COUNT(*) AS bronze_rows FROM bronze.table_a GROUP BY __filepath__) b
LEFT JOIN (SELECT source_file, COUNT(*) AS silver_rows FROM silver.table_a_clean GROUP BY source_file) s
  ON s.source_file = b.__filepath__
ORDER BY file;

In [ ]:
%%sql
-- (c) Which bronze rows the silver constraints dropped (mirrors the three CHECK conditions).
SELECT element_at(split(__filepath__, '/'), -1) AS file, id, segment, value,
       CASE WHEN id IS NULL THEN 'id_not_null'
            WHEN CAST(value AS DECIMAL(10,2)) < 0 THEN 'non_negative_value'
            ELSE 'known_segment' END AS violated_constraint
FROM bronze.table_a
WHERE id IS NULL
   OR CAST(value AS DECIMAL(10,2)) < 0
   OR segment NOT IN ('A', 'B', 'C')
ORDER BY file, violated_constraint;

In [ ]:
%%sql
-- (d) Schema of bronze: after file_003, attribute_b should appear (DYNAMIC schema mode).
--   DESCRIBE on an MLV is standard Spark SQL on a Delta table; not shown explicitly in MLV docs.
DESCRIBE TABLE bronze.table_a;

In [ ]:
%%sql
-- (e) After file_003 only: attribute_b population per file (NULL for files that predate the column).
SELECT element_at(split(__filepath__, '/'), -1) AS file,
       COUNT(*) AS rows, COUNT(attribute_b) AS rows_with_attribute_b
FROM bronze.table_a
GROUP BY __filepath__
ORDER BY file;

In [ ]:
%%sql
-- (f) Gold
SELECT * FROM gold.value_by_segment ORDER BY segment;

## Refresh between files
Upload the next file to the same folder, then refresh. Preferred: Lakehouse > **Materialized lake views > Manage > Run** (ad-hoc, *Refresh with dependant lineage*) so the run shows *files processed / rows added*. Notebook fallback below (FULL, in dependency order). Then re-run section 4.

In [ ]:
%%sql
-- 06_refresh_full_optional.sql - notebook fallback if you can't use the lineage "Run" button.
-- Documented syntax: REFRESH MATERIALIZED LAKE VIEW [workspace.lakehouse.schema].MLV_Identifier FULL
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/refresh-materialized-lake-view#run-a-one-time-full-refresh-with-sql
-- and for file views "REFRESH MATERIALIZED LAKE VIEW <bronze view> FULL;" (blog + Spark SQL reference tip).
-- Docs recommend lineage/scheduled refresh for ongoing runs ("don't orchestrate ... from a notebook").
-- A notebook refresh does NOT cascade; run in dependency order yourself.
-- FULL on bronze reprocesses the current folder as a snapshot, so no duplicate rows.

REFRESH MATERIALIZED LAKE VIEW bronze.table_a FULL;

In [ ]:
%%sql
REFRESH MATERIALIZED LAKE VIEW silver.table_a_clean FULL;

In [ ]:
%%sql
REFRESH MATERIALIZED LAKE VIEW gold.value_by_segment FULL;
-- after step 05 only:
-- REFRESH MATERIALIZED LAKE VIEW gold.value_by_attribute_b FULL

## 5. After file_003: evolve silver deliberately + new gold view

In [ ]:
%%sql
-- 05_after_file3_silver_v2.sql  - run once bronze shows attribute_b (after the file_003 refresh).
-- Point of the step: bronze evolves automatically (DYNAMIC); silver is an explicit contract you
-- change deliberately. "To modify the definition ... use the CREATE OR REPLACE command":
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/create-materialized-lake-view#update-a-materialized-lake-view
-- OR REPLACE can't be combined with IF NOT EXISTS (Arguments table, same page).
-- UNVERIFIED: whether replacing silver while gold.value_by_segment depends on it keeps the lineage
-- intact without recreating gold. If gold errors on its next run, DROP gold.value_by_segment
-- and re-run 03_gold_value_by_segment.sql.

CREATE OR REPLACE MATERIALIZED LAKE VIEW silver.table_a_clean
(
    CONSTRAINT id_not_null         CHECK (id IS NOT NULL)              ON MISMATCH DROP,
    CONSTRAINT non_negative_value  CHECK (value >= 0)                  ON MISMATCH DROP,
    CONSTRAINT known_segment       CHECK (segment IN ('A', 'B', 'C'))  ON MISMATCH DROP
)
COMMENT "Silver v2: adds attribute_b (introduced in file_003)"
TBLPROPERTIES (delta.enableChangeDataFeed=true)
AS SELECT
    CAST(id AS INT)                AS id,
    segment,
    attribute_a,
    CAST(value AS DECIMAL(10,2))   AS value,
    attribute_b,
    __filepath__                   AS source_file
FROM bronze.table_a;

In [ ]:
%%sql
-- New gold view that only makes sense once the column exists.
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS gold.value_by_attribute_b
COMMENT "Gold: row count and total value by attribute_b"
AS SELECT
    COALESCE(attribute_b, '(none)') AS attribute_b,
    COUNT(*)                        AS row_count,
    SUM(value)                      AS total_value
FROM silver.table_a_clean
GROUP BY COALESCE(attribute_b, '(none)');

In [ ]:
%%sql
SELECT * FROM gold.value_by_attribute_b ORDER BY attribute_b;

## 99. Cleanup (run only when done)

In [ ]:
%%sql
-- 99_cleanup.sql - drop downstream first. Syntax: DROP MATERIALIZED LAKE VIEW MLV_Identifier--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/create-materialized-lake-view#delete-a-materialized-lake-view
-- Also delete any refresh schedule you created (Manage schedules > trash icon):
--   https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/schedule-lineage-run#manage-schedules
-- Do not touch the _mlv_system schema (overview page, "Don't alter the _mlv_system schema").
-- These names match the objects already in demo_ws (demo_mlv_lh), so this also drops those.

DROP MATERIALIZED LAKE VIEW gold.value_by_attribute_b;

In [ ]:
%%sql
DROP MATERIALIZED LAKE VIEW gold.value_by_segment;

In [ ]:
%%sql
DROP MATERIALIZED LAKE VIEW silver.table_a_clean;

In [ ]:
%%sql
DROP MATERIALIZED LAKE VIEW bronze.table_a;
-- Optional, if the schemas were created only for this demo (standard Spark SQL, not from MLV docs):
-- DROP SCHEMA gold; DROP SCHEMA silver; DROP SCHEMA bronze
-- Finally delete the Files/mlv_demo/table_a folder in the Lakehouse explorer.